In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
file_path = "../raw_data/online_retail_II.xlsx"

excel_file = pd.ExcelFile(file_path)

print(excel_file.sheet_names)

['Year 2009-2010', 'Year 2010-2011']


In [3]:
# inspecting first 5 rows in each 

for sheet in excel_file.sheet_names:
    print(f"\n{'='*60}")
    print(f"Sheet: {sheet}")
    print(f"{'='*60}")
    
    sample = pd.read_excel(
        file_path,
        sheet_name=sheet,
        nrows=5
    )
    
    print("Columns:")
    print(sample.columns.tolist())
    
    print("\nFirst 5 rows:")
    display(sample)


Sheet: Year 2009-2010
Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

First 5 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom



Sheet: Year 2010-2011
Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

First 5 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [4]:
# Loading a small sample from each sheet for data quality inspection

for sheet in excel_file.sheet_names:
    df_sample = pd.read_excel(
        file_path,
        sheet_name=sheet,
        nrows=10000
    )

    print(f"\n{'='*60}")
    print(f"DATA QUALITY CHECK: {sheet}")
    print(f"{'='*60}")

    print("\nData types:")
    print(df_sample.dtypes)

    print("\nMissing values:")
    print(df_sample.isnull().sum())

    print("\nDuplicate rows:")
    print(df_sample.duplicated().sum())


DATA QUALITY CHECK: Year 2009-2010

Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Missing values:
Invoice           0
StockCode         0
Description      16
Quantity          0
InvoiceDate       0
Price             0
Customer ID    2805
Country           0
dtype: int64

Duplicate rows:
94

DATA QUALITY CHECK: Year 2010-2011

Data types:
Invoice                object
StockCode              object
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Missing values:
Invoice           0
StockCode         0
Description      42
Quantity          0
InvoiceDate       0
Price             0
Customer ID    2291
Country           0
dtype: 

In [5]:
# Loading both sheets completely

df_2009_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

df_2010_2011 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011"
)

print("2009-2010 shape:", df_2009_2010.shape)
print("2010-2011 shape:", df_2010_2011.shape)

print("\n2009-2010 missing values:")
print(df_2009_2010.isnull().sum())

print("\n2010-2011 missing values:")
print(df_2010_2011.isnull().sum())

print("\n2009-2010 duplicate rows:",
      df_2009_2010.duplicated().sum())

print("2010-2011 duplicate rows:",
      df_2010_2011.duplicated().sum())

2009-2010 shape: (525461, 8)
2010-2011 shape: (541910, 8)

2009-2010 missing values:
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

2010-2011 missing values:
Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

2009-2010 duplicate rows: 6865
2010-2011 duplicate rows: 5268


In [6]:
# Combineing both yearly datasets

df = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

print("Combined dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nTotal missing values:")
print(df.isnull().sum())

print("\nTotal duplicate rows:")
print(df.duplicated().sum())

Combined dataset shape: (1067371, 8)

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Total missing values:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Total duplicate rows:
34335


In [7]:
# Investigateing cancellations, negative quantities, and unusual prices

print("=" * 60)
print("CANCELLATION ANALYSIS")
print("=" * 60)

cancelled = df["Invoice"].astype(str).str.startswith("C")

print("Cancellation rows:", cancelled.sum())
print("Cancellation invoices:", df.loc[cancelled, "Invoice"].nunique())

print("\n" + "=" * 60)
print("QUANTITY ANALYSIS")
print("=" * 60)

print("Negative quantity rows:", (df["Quantity"] < 0).sum())
print("Zero quantity rows:", (df["Quantity"] == 0).sum())
print("Minimum quantity:", df["Quantity"].min())
print("Maximum quantity:", df["Quantity"].max())

print("\n" + "=" * 60)
print("PRICE ANALYSIS")
print("=" * 60)

print("Negative price rows:", (df["Price"] < 0).sum())
print("Zero price rows:", (df["Price"] == 0).sum())
print("Minimum price:", df["Price"].min())
print("Maximum price:", df["Price"].max())

print("\n" + "=" * 60)
print("DATE ANALYSIS")
print("=" * 60)

print("Earliest transaction:", df["InvoiceDate"].min())
print("Latest transaction:", df["InvoiceDate"].max())

CANCELLATION ANALYSIS
Cancellation rows: 19494
Cancellation invoices: 8292

QUANTITY ANALYSIS
Negative quantity rows: 22950
Zero quantity rows: 0
Minimum quantity: -80995
Maximum quantity: 80995

PRICE ANALYSIS
Negative price rows: 5
Zero price rows: 6202
Minimum price: -53594.36
Maximum price: 38970.0

DATE ANALYSIS
Earliest transaction: 2009-12-01 07:45:00
Latest transaction: 2011-12-09 12:50:00


In [8]:
# Investigateing the relationship between cancellations and negative quantities

negative_qty = df["Quantity"] < 0
cancelled = df["Invoice"].astype(str).str.startswith("C")

print("=" * 60)
print("NEGATIVE QUANTITY vs CANCELLATION")
print("=" * 60)

print("Negative quantity rows:", negative_qty.sum())
print("Negative quantity + cancellation:", (negative_qty & cancelled).sum())
print("Negative quantity WITHOUT cancellation:", (negative_qty & ~cancelled).sum())

print("\nCancellation rows with positive quantity:",
      (cancelled & (df["Quantity"] > 0)).sum())

print("\n" + "=" * 60)
print("NEGATIVE QUANTITY WITH CUSTOMER IDs")
print("=" * 60)

print(
    "Negative quantity rows with Customer ID:",
    df.loc[negative_qty, "Customer ID"].notna().sum()
)

print(
    "Negative quantity rows WITHOUT Customer ID:",
    df.loc[negative_qty, "Customer ID"].isna().sum()
)

print("\n" + "=" * 60)
print("LARGEST NEGATIVE QUANTITIES")
print("=" * 60)

display(
    df.loc[negative_qty]
      .sort_values("Quantity")
      [["Invoice", "StockCode", "Description",
        "Quantity", "InvoiceDate", "Price",
        "Customer ID", "Country"]]
      .head(10)
)

NEGATIVE QUANTITY vs CANCELLATION
Negative quantity rows: 22950
Negative quantity + cancellation: 19493
Negative quantity WITHOUT cancellation: 3457

Cancellation rows with positive quantity: 1

NEGATIVE QUANTITY WITH CUSTOMER IDs
Negative quantity rows with Customer ID: 18744
Negative quantity rows WITHOUT Customer ID: 4206

LARGEST NEGATIVE QUANTITIES


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom
750990,556690,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.00,NaN,United Kingdom
303996,519017,22759,NaN,-9600,2010-08-13 09:14:00,0.00,NaN,United Kingdom
750991,556691,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.00,NaN,United Kingdom
507225,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom
529729,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom
156488,504311,22197,NaN,-9200,2010-04-12 14:39:00,0.00,NaN,United Kingdom
750989,556687,23003,Printing smudges/thrown away,-9058,2011-06-14 10:36:00,0.00,NaN,United Kingdom
194372,507913,10120,Zebra invcing error,-9000,2010-05-11 17:16:00,0.00,NaN,United Kingdom


In [9]:
# Customer and transaction profile

print("=" * 60)
print("CUSTOMER PROFILE")
print("=" * 60)

print("Unique Customer IDs:",
      df["Customer ID"].nunique())

print("Missing Customer IDs:",
      df["Customer ID"].isna().sum())

print("Customers with IDs:",
      df["Customer ID"].notna().sum())

print("\n" + "=" * 60)
print("INVOICE PROFILE")
print("=" * 60)

print("Unique invoices:",
      df["Invoice"].nunique())

print("Unique cancellation invoices:",
      df.loc[
          df["Invoice"].astype(str).str.startswith("C"),
          "Invoice"
      ].nunique())

print("\n" + "=" * 60)
print("COUNTRY PROFILE")
print("=" * 60)

print("Unique countries:",
      df["Country"].nunique())

print("\nTop 15 countries by transaction rows:")

display(
    df["Country"]
      .value_counts()
      .head(15)
      .to_frame("Transaction_Rows")
)

CUSTOMER PROFILE
Unique Customer IDs: 5942
Missing Customer IDs: 243007
Customers with IDs: 824364

INVOICE PROFILE
Unique invoices: 53628
Unique cancellation invoices: 8292

COUNTRY PROFILE
Unique countries: 43

Top 15 countries by transaction rows:


,Transaction_Rows
Country,
United Kingdom,981330
EIRE,17866
Germany,17624
France,14330
Netherlands,5140
Spain,3811
Switzerland,3189
Belgium,3123
Portugal,2620


In [10]:
# Classify transaction records for cleaning analysis

df["Is_Cancellation"] = (
    df["Invoice"]
    .astype(str)
    .str.startswith("C")
)

df["Is_Negative_Quantity"] = df["Quantity"] < 0

df["Is_Zero_Price"] = df["Price"] == 0

df["Is_Negative_Price"] = df["Price"] < 0

df["Has_Customer_ID"] = df["Customer ID"].notna()

print("=" * 60)
print("TRANSACTION CLASSIFICATION")
print("=" * 60)

print("Total rows:", len(df))

print("\nCancellation rows:",
      df["Is_Cancellation"].sum())

print("Negative quantity rows:",
      df["Is_Negative_Quantity"].sum())

print("Zero-price rows:",
      df["Is_Zero_Price"].sum())

print("Negative-price rows:",
      df["Is_Negative_Price"].sum())

print("\nRows with Customer ID:",
      df["Has_Customer_ID"].sum())

print("Rows without Customer ID:",
      (~df["Has_Customer_ID"]).sum())

print("\n" + "=" * 60)
print("POTENTIAL PURCHASE TRANSACTIONS")
print("=" * 60)

potential_purchase = (
    (~df["Is_Cancellation"]) &
    (df["Quantity"] > 0) &
    (df["Price"] > 0) &
    (df["Has_Customer_ID"])
)

print(
    "Rows meeting all purchase criteria:",
    potential_purchase.sum()
)

print(
    "Unique customers in potential purchases:",
    df.loc[potential_purchase, "Customer ID"].nunique()
)

print(
    "Unique invoices in potential purchases:",
    df.loc[potential_purchase, "Invoice"].nunique()
)

TRANSACTION CLASSIFICATION
Total rows: 1067371

Cancellation rows: 19494
Negative quantity rows: 22950
Zero-price rows: 6202
Negative-price rows: 5

Rows with Customer ID: 824364
Rows without Customer ID: 243007

POTENTIAL PURCHASE TRANSACTIONS
Rows meeting all purchase criteria: 805549
Unique customers in potential purchases: 5878
Unique invoices in potential purchases: 36969


In [11]:
# Analyzeing records excluded from our potential purchase definition

excluded = ~potential_purchase

print("=" * 60)
print("EXCLUDED RECORDS")
print("=" * 60)

print("Total excluded rows:", excluded.sum())

print("\nExcluded rows by reason:")

print(
    "Cancellation:",
    (excluded & df["Is_Cancellation"]).sum()
)

print(
    "Negative quantity:",
    (excluded & (df["Quantity"] < 0)).sum()
)

print(
    "Zero price:",
    (excluded & (df["Price"] == 0)).sum()
)

print(
    "Negative price:",
    (excluded & (df["Price"] < 0)).sum()
)

print(
    "Missing Customer ID:",
    (excluded & df["Customer ID"].isna()).sum()
)

print("\n" + "=" * 60)
print("CUSTOMER TRANSACTIONS EXCLUDED")
print("=" * 60)

customer_excluded = (
    excluded &
    df["Customer ID"].notna()
)

print(
    "Rows with Customer ID but excluded:",
    customer_excluded.sum()
)

print(
    "Customers affected:",
    df.loc[customer_excluded, "Customer ID"].nunique()
)

print("\nSample of excluded customer records:")

display(
    df.loc[
        customer_excluded,
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ].head(15)
)

EXCLUDED RECORDS
Total excluded rows: 261822

Excluded rows by reason:
Cancellation: 19494
Negative quantity: 22950
Zero price: 6202
Negative price: 5
Missing Customer ID: 243007

CUSTOMER TRANSACTIONS EXCLUDED
Rows with Customer ID but excluded: 18815
Customers affected: 2582

Sample of excluded customer records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


In [12]:
# Createing mutually exclusive transaction categories

conditions = [
    df["Customer ID"].isna(),
    df["Is_Cancellation"],
    df["Quantity"] < 0,
    df["Price"] <= 0
]

choices = [
    "Missing_Customer_ID",
    "Cancellation_Return",
    "Negative_Quantity",
    "Non_Positive_Price"
]

df["Transaction_Type"] = np.select(
    conditions,
    choices,
    default="Purchase"
)

print("=" * 60)
print("TRANSACTION TYPE BREAKDOWN")
print("=" * 60)

transaction_summary = (
    df["Transaction_Type"]
    .value_counts()
    .to_frame("Rows")
)

transaction_summary["Percentage"] = (
    transaction_summary["Rows"] / len(df) * 100
).round(2)

display(transaction_summary)

TRANSACTION TYPE BREAKDOWN


,Rows,Percentage
Transaction_Type,,
Purchase,805549,75.47
Missing_Customer_ID,243007,22.77
Cancellation_Return,18744,1.76
Non_Positive_Price,71,0.01


In [13]:
# Createing the cleaned customer purchase dataset

customer_purchases = df[
    (df["Transaction_Type"] == "Purchase") &
    (df["Customer ID"].notna())
].copy()

print("=" * 60)
print("CLEANED CUSTOMER PURCHASE DATASET")
print("=" * 60)

print("Rows:", len(customer_purchases))
print("Columns:", customer_purchases.shape[1])
print("Unique customers:", customer_purchases["Customer ID"].nunique())
print("Unique invoices:", customer_purchases["Invoice"].nunique())

print("\nRemaining missing values:")
print(customer_purchases.isnull().sum())

print("\nRemaining duplicate rows:",
      customer_purchases.duplicated().sum())

CLEANED CUSTOMER PURCHASE DATASET
Rows: 805549
Columns: 14
Unique customers: 5878
Unique invoices: 36969

Remaining missing values:
Invoice                 0
StockCode               0
Description             0
Quantity                0
InvoiceDate             0
Price                   0
Customer ID             0
Country                 0
Is_Cancellation         0
Is_Negative_Quantity    0
Is_Zero_Price           0
Is_Negative_Price       0
Has_Customer_ID         0
Transaction_Type        0
dtype: int64

Remaining duplicate rows: 26124


In [14]:
# Investigateing duplicate purchase records

duplicate_mask = customer_purchases.duplicated(keep=False)

duplicate_records = customer_purchases[
    duplicate_mask
].sort_values(
    ["Invoice", "StockCode", "InvoiceDate", "Customer ID"]
)

print("=" * 60)
print("DUPLICATE INVESTIGATION")
print("=" * 60)

print("Duplicate rows:", duplicate_mask.sum())

print(
    "Duplicate invoice + stock code combinations:",
    duplicate_records[["Invoice", "StockCode"]]
    .duplicated()
    .sum()
)

print("\nSample duplicate records:")

display(
    duplicate_records[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ].head(20)
)

DUPLICATE INVESTIGATION
Duplicate rows: 50836
Duplicate invoice + stock code combinations: 26406

Sample duplicate records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [15]:
# Checking how many duplicate groups exist

duplicate_groups = (
    customer_purchases
    .groupby(
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="Count")
)

print("Total unique transaction groups:", len(duplicate_groups))

print("\nDuplicate groups (appearing more than once):")

display(
    duplicate_groups[
        duplicate_groups["Count"] > 1
    ]
    .sort_values("Count", ascending=False)
    .head(20)
)

Total unique transaction groups: 779425

Duplicate groups (appearing more than once):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Count
533099,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,20
533098,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,12
394184,537224,70007,HI TEC ALPINE HAND WARMER,1,2010-12-05 16:24:00,1.65,13174.0,United Kingdom,10
689834,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102.0,United Kingdom,8
52117,496431,84826,ASSTD DESIGN 3D PAPER STICKERS,1,2010-02-01 12:30:00,0.85,16415.0,United Kingdom,6
391968,537051,22730,ALARM CLOCK BAKELIKE IVORY,1,2010-12-05 11:12:00,3.75,15708.0,United Kingdom,6
387270,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom,6
685696,572344,M,Manual,48,2011-10-24 10:43:00,1.50,14607.0,United Kingdom,6
99688,502660,17021,NAMASTE SWAGAT INCENSE,6,2010-03-25 17:18:00,0.30,13187.0,United Kingdom,6
390056,536796,21967,PACK OF 12 SKULL TISSUES,1,2010-12-02 15:46:00,0.29,15574.0,United Kingdom,6


In [16]:
# Removeing exact duplicate purchase records

before_dedup = len(customer_purchases)

customer_purchases = customer_purchases.drop_duplicates(
    keep="first"
).copy()

after_dedup = len(customer_purchases)

duplicates_removed = before_dedup - after_dedup

print("=" * 60)
print("DUPLICATE REMOVAL")
print("=" * 60)

print("Rows before:", before_dedup)
print("Rows after:", after_dedup)
print("Duplicates removed:", duplicates_removed)

print("\nRemaining duplicate rows:",
      customer_purchases.duplicated().sum())

print("\nUnique customers:",
      customer_purchases["Customer ID"].nunique())

print("Unique invoices:",
      customer_purchases["Invoice"].nunique())

DUPLICATE REMOVAL
Rows before: 805549
Rows after: 779425
Duplicates removed: 26124

Remaining duplicate rows: 0

Unique customers: 5878
Unique invoices: 36969


In [17]:
# Standardizeing data types

customer_purchases["Invoice"] = (
    customer_purchases["Invoice"]
    .astype(str)
    .str.strip()
)

customer_purchases["StockCode"] = (
    customer_purchases["StockCode"]
    .astype(str)
    .str.strip()
)

customer_purchases["Description"] = (
    customer_purchases["Description"]
    .astype(str)
    .str.strip()
)

customer_purchases["Quantity"] = (
    pd.to_numeric(
        customer_purchases["Quantity"],
        errors="coerce"
    )
    .astype("int64")
)

customer_purchases["InvoiceDate"] = pd.to_datetime(
    customer_purchases["InvoiceDate"],
    errors="coerce"
)

customer_purchases["Price"] = pd.to_numeric(
    customer_purchases["Price"],
    errors="coerce"
)

customer_purchases["Customer ID"] = (
    pd.to_numeric(
        customer_purchases["Customer ID"],
        errors="coerce"
    )
    .astype("int64")
)

customer_purchases["Country"] = (
    customer_purchases["Country"]
    .astype(str)
    .str.strip()
)

print("=" * 60)
print("DATA TYPES AFTER STANDARDIZATION")
print("=" * 60)

print(customer_purchases.dtypes)

DATA TYPES AFTER STANDARDIZATION
Invoice                            str
StockCode                          str
Description                        str
Quantity                         int64
InvoiceDate             datetime64[us]
Price                          float64
Customer ID                      int64
Country                            str
Is_Cancellation                   bool
Is_Negative_Quantity              bool
Is_Zero_Price                     bool
Is_Negative_Price                 bool
Has_Customer_ID                   bool
Transaction_Type                   str
dtype: object


In [18]:
# Verifying data quality after type conversion

print("=" * 60)
print("POST-CONVERSION QUALITY CHECK")
print("=" * 60)

print("Missing values:")
print(customer_purchases.isnull().sum())

print("\nDuplicate rows:",
      customer_purchases.duplicated().sum())

print("\nNegative quantities:",
      (customer_purchases["Quantity"] < 0).sum())

print("Non-positive prices:",
      (customer_purchases["Price"] <= 0).sum())

print("\nDate range:")
print(customer_purchases["InvoiceDate"].min())
print(customer_purchases["InvoiceDate"].max())

POST-CONVERSION QUALITY CHECK
Missing values:
Invoice                 0
StockCode               0
Description             0
Quantity                0
InvoiceDate             0
Price                   0
Customer ID             0
Country                 0
Is_Cancellation         0
Is_Negative_Quantity    0
Is_Zero_Price           0
Is_Negative_Price       0
Has_Customer_ID         0
Transaction_Type        0
dtype: int64

Duplicate rows: 0

Negative quantities: 0
Non-positive prices: 0

Date range:
2009-12-01 07:45:00
2011-12-09 12:50:00


In [19]:
# Calculateing transaction-level revenue

customer_purchases["Revenue"] = (
    customer_purchases["Quantity"] *
    customer_purchases["Price"]
)

print("=" * 60)
print("REVENUE CALCULATION")
print("=" * 60)

print("Total revenue:",
      customer_purchases["Revenue"].sum())

print("Average transaction-line revenue:",
      customer_purchases["Revenue"].mean())

print("Minimum transaction-line revenue:",
      customer_purchases["Revenue"].min())

print("Maximum transaction-line revenue:",
      customer_purchases["Revenue"].max())

print("\nRevenue statistics:")
print(
    customer_purchases["Revenue"]
    .describe()
)

REVENUE CALCULATION
Total revenue: 17374804.268
Average transaction-line revenue: 22.291823161946304
Minimum transaction-line revenue: 0.001
Maximum transaction-line revenue: 168469.6

Revenue statistics:
count    779425.000000
mean         22.291823
std         227.427075
min           0.001000
25%           4.950000
50%          12.480000
75%          19.800000
max      168469.600000
Name: Revenue, dtype: float64


In [20]:
# Investigateing unusually high-revenue purchase lines

print("=" * 60)
print("TOP 20 TRANSACTION LINES BY REVENUE")
print("=" * 60)

top_revenue = (
    customer_purchases
    .sort_values("Revenue", ascending=False)
    [
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country",
            "Revenue"
        ]
    ]
    .head(20)
)

display(top_revenue)

TOP 20 TRANSACTION LINES BY REVENUE


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom,168469.60
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,United Kingdom,77183.60
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098,United Kingdom,38970.00
432176,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,2010-11-04 11:36:00,1.69,15838,United Kingdom,15818.40
135013,502263,M,Manual,1,2010-03-23 15:22:00,10953.50,12918,United Kingdom,10953.50
358639,524159,M,Manual,1,2010-09-27 16:12:00,10468.80,14063,United Kingdom,10468.80
74356,496115,M,Manual,1,2010-01-29 11:04:00,8985.60,17949,United Kingdom,8985.60
228042,511465,15044A,PINK PAPER PARASOL,3500,2010-06-08 12:59:00,2.55,18008,United Kingdom,8925.00
698843,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029,United Kingdom,8142.75
873786,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,2011-09-20 11:05:00,5.06,17450,United Kingdom,7144.72


In [21]:
# Revenue percentile thresholds

print("=" * 60)
print("REVENUE PERCENTILES")
print("=" * 60)

print(
    customer_purchases["Revenue"]
    .quantile([0.90, 0.95, 0.99, 0.995, 0.999])
)

REVENUE PERCENTILES
0.900     35.70
0.950     67.50
0.990    203.52
0.995    306.00
0.999    864.00
Name: Revenue, dtype: float64


In [22]:
# Investigateing unusually high-revenue purchase lines

top_revenue = (
    customer_purchases
    .sort_values("Revenue", ascending=False)
    [[
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country",
        "Revenue"
    ]]
    .head(20)
)

display(top_revenue)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom,168469.60
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,United Kingdom,77183.60
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098,United Kingdom,38970.00
432176,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,2010-11-04 11:36:00,1.69,15838,United Kingdom,15818.40
135013,502263,M,Manual,1,2010-03-23 15:22:00,10953.50,12918,United Kingdom,10953.50
358639,524159,M,Manual,1,2010-09-27 16:12:00,10468.80,14063,United Kingdom,10468.80
74356,496115,M,Manual,1,2010-01-29 11:04:00,8985.60,17949,United Kingdom,8985.60
228042,511465,15044A,PINK PAPER PARASOL,3500,2010-06-08 12:59:00,2.55,18008,United Kingdom,8925.00
698843,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029,United Kingdom,8142.75
873786,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,2011-09-20 11:05:00,5.06,17450,United Kingdom,7144.72


In [23]:
# Investigateing products involved in the highest-revenue transactions

top_products = customer_purchases.loc[
    customer_purchases["Revenue"].nlargest(10).index,
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "Revenue"]
]

for stock_code in top_products["StockCode"].unique():
    print("\n" + "=" * 80)
    print("StockCode:", stock_code)

    product_history = (
        customer_purchases[
            customer_purchases["StockCode"] == stock_code
        ][
            ["Invoice", "StockCode", "Description", "Quantity", "Price", "Revenue"]
        ]
        .sort_values("Quantity", ascending=False)
        .head(10)
    )

    display(product_history)


StockCode: 23843


,Invoice,StockCode,Description,Quantity,Price,Revenue
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.6



StockCode: 23166


,Invoice,StockCode,Description,Quantity,Price,Revenue
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60
810828,561901,23166,MEDIUM CERAMIC TOP STORAGE JAR,288,1.25,360.00
719923,553607,23166,MEDIUM CERAMIC TOP STORAGE JAR,240,1.04,249.60
801974,561051,23166,MEDIUM CERAMIC TOP STORAGE JAR,144,1.04,149.76
727031,554307,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84
753621,556917,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84
1051842,580665,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84
830390,563614,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84
712231,552882,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84
1011236,577669,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84



StockCode: 22502


...

,Invoice,StockCode,Description,Quantity,Price,Revenue
228042,511465,15044A,PINK PAPER PARASOL,3500,2.55,8925.0
172966,505747,15044A,PINK PAPER PARASOL,200,2.55,510.0
416117,529146,15044A,PINK PAPER PARASOL,100,2.55,255.0
262881,514729,15044A,PINK PAPER PARASOL,50,2.55,127.5
243214,512938,15044A,PINK PAPER PARASOL,50,2.55,127.5
634542,545582,15044A,PINK PAPER PARASOL,50,2.55,127.5
255801,514082,15044A,PINK PAPER PARASOL,50,2.55,127.5
106296,499521,15044A,PINK PAPER PARASOL,50,2.55,127.5
187859,507235,15044A,PINK PAPER PARASOL,24,2.55,61.2
596988,542142,15044A,PINK PAPER PARASOL,18,2.95,53.1



StockCode: POST


,Invoice,StockCode,Description,Quantity,Price,Revenue
548047,538175,POST,POSTAGE,21,18.00,378.0
587079,541430,POST,POSTAGE,18,18.00,324.0
490157,535321,POST,POSTAGE,18,15.00,270.0
451391,532305,POST,POSTAGE,18,18.00,324.0
224971,511117,POST,POSTAGE,16,10.95,175.2
659564,547822,POST,POSTAGE,16,18.00,288.0
154014,503985,POST,POSTAGE,15,18.00,270.0
460432,532975,POST,POSTAGE,14,18.00,252.0
502271,536239,POST,POSTAGE,14,18.00,252.0
76618,496340,POST,POSTAGE,13,18.00,234.0



StockCode: 23243


,Invoice,StockCode,Description,Quantity,Price,Revenue
873786,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,5.06,7144.72
830120,563590,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,352,5.15,1812.80
753633,556917,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,240,4.15,996.00
879353,567852,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,216,3.39,732.24
755635,557125,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,152,4.15,630.80
1034247,579271,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,139,5.13,713.07
810850,561902,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,96,4.95,475.20
873730,567385,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,72,4.15,298.80
989804,576088,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,50,5.17,258.50
783621,559554,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,48,4.15,199.20


In [24]:
# Finding transactions with unusually large quantities

quantity_outliers = (
    customer_purchases[
        customer_purchases["Quantity"] > 1000
    ]
    [[
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Revenue",
        "Customer ID",
        "InvoiceDate"
    ]]
    .sort_values("Quantity", ascending=False)
)

print("Transactions with Quantity > 1,000:", len(quantity_outliers))
display(quantity_outliers.head(30))

Transactions with Quantity > 1,000: 295


,Invoice,StockCode,Description,Quantity,Price,Revenue,Customer ID,InvoiceDate
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,16446,2011-12-09 09:15:00
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,12346,2011-01-18 10:01:00
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.20,13902,2010-02-15 11:57:00
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.00,13902,2010-03-17 13:09:00
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.00,13902,2010-03-17 13:09:00
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.40,13902,2010-03-17 13:09:00
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.00,13902,2010-03-17 13:09:00
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.00,17940,2010-03-23 15:36:00
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.00,17940,2010-03-23 15:36:00
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,0.25,2500.00,17940,2010-03-23 15:36:00


In [25]:
# Detecting extreme quantity outliers using the IQR method

Q1 = customer_purchases["Quantity"].quantile(0.25)
Q3 = customer_purchases["Quantity"].quantile(0.75)

IQR = Q3 - Q1
upper_quantity_limit = Q3 + (1.5 * IQR)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Upper quantity limit:", upper_quantity_limit)

quantity_outliers_iqr = customer_purchases[
    customer_purchases["Quantity"] > upper_quantity_limit
].copy()

print("\nNumber of quantity outliers:", len(quantity_outliers_iqr))
print("Percentage of purchase lines:", 
      round(len(quantity_outliers_iqr) / len(customer_purchases) * 100, 2), "%")

display(
    quantity_outliers_iqr[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Revenue",
            "Customer ID",
            "InvoiceDate"
        ]
    ]
    .sort_values("Quantity", ascending=False)
    .head(30)
)

Q1: 2.0
Q3: 12.0
IQR: 10.0
Upper quantity limit: 27.0

Number of quantity outliers: 51119
Percentage of purchase lines: 6.56 %


,Invoice,StockCode,Description,Quantity,Price,Revenue,Customer ID,InvoiceDate
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,16446,2011-12-09 09:15:00
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,12346,2011-01-18 10:01:00
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.20,13902,2010-02-15 11:57:00
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.00,13902,2010-03-17 13:09:00
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.00,13902,2010-03-17 13:09:00
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.40,13902,2010-03-17 13:09:00
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.00,13902,2010-03-17 13:09:00
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.00,17940,2010-03-23 15:36:00
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.00,17940,2010-03-23 15:36:00
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,0.25,2500.00,17940,2010-03-23 15:36:00


In [26]:
# Measureing revenue contribution of high-quantity transactions

quantity_thresholds = [100, 500, 1000, 5000, 10000]

total_revenue = customer_purchases["Revenue"].sum()

for threshold in quantity_thresholds:
    subset = customer_purchases[
        customer_purchases["Quantity"] > threshold
    ]

    revenue = subset["Revenue"].sum()
    percentage = (revenue / total_revenue) * 100

    print(
        f"Quantity > {threshold:>5}: "
        f"{len(subset):>6,} rows | "
        f"Revenue = £{revenue:,.2f} | "
        f"Revenue share = {percentage:.2f}%"
    )

Quantity >   100:  9,812 rows | Revenue = £3,380,881.99 | Revenue share = 19.46%
Quantity >   500:    932 rows | Revenue = £1,014,852.69 | Revenue share = 5.84%
Quantity >  1000:    295 rows | Revenue = £568,636.51 | Revenue share = 3.27%
Quantity >  5000:     30 rows | Revenue = £294,158.80 | Revenue share = 1.69%
Quantity > 10000:      7 rows | Revenue = £252,682.80 | Revenue share = 1.45%


In [27]:
# Inspecting the most extreme quantity transactions

extreme_quantity = (
    customer_purchases[
        customer_purchases["Quantity"] > 10000
    ][
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Revenue",
            "Customer ID",
            "Country",
            "InvoiceDate"
        ]
    ]
    .sort_values("Quantity", ascending=False)
)

display(extreme_quantity)

,Invoice,StockCode,Description,Quantity,Price,Revenue,Customer ID,Country,InvoiceDate
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.6,16446,United Kingdom,2011-12-09 09:15:00
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.6,12346,United Kingdom,2011-01-18 10:01:00
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.2,13902,Denmark,2010-02-15 11:57:00
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.0,13902,Denmark,2010-03-17 13:09:00
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.0,13902,Denmark,2010-03-17 13:09:00
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.4,13902,Denmark,2010-03-17 13:09:00
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.0,13902,Denmark,2010-03-17 13:09:00


In [28]:
# Checking whether the extreme transactions have matching cancellation records

extreme_invoices = extreme_quantity["Invoice"].tolist()

cancellation_matches = df[
    df["Invoice"].astype(str).str.startswith("C") &
    df["Invoice"].astype(str).str[1:].isin(extreme_invoices)
][[
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "Price",
    "InvoiceDate",
    "Customer ID",
    "Country"
]].copy()

display(cancellation_matches)

,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Customer ID,Country


In [29]:
# Checking whether extreme purchases have matching cancellation/return lines
# based on Customer ID + StockCode + opposite Quantity

extreme_check = extreme_quantity[
    extreme_quantity["Quantity"] > 10000
].copy()

cancellations = df[
    df["Is_Cancellation"] &
    df["Customer ID"].notna()
].copy()

matches = []

for _, purchase in extreme_check.iterrows():

    possible_matches = cancellations[
        (cancellations["Customer ID"] == purchase["Customer ID"]) &
        (cancellations["StockCode"].astype(str) == str(purchase["StockCode"])) &
        (cancellations["Quantity"] == -purchase["Quantity"])
    ].copy()

    for _, cancel in possible_matches.iterrows():
        matches.append({
            "Purchase_Invoice": purchase["Invoice"],
            "Purchase_Quantity": purchase["Quantity"],
            "Purchase_Revenue": purchase["Revenue"],
            "Purchase_Date": purchase["InvoiceDate"],
            "Cancellation_Invoice": cancel["Invoice"],
            "Cancellation_Quantity": cancel["Quantity"],
            "Cancellation_Date": cancel["InvoiceDate"],
            "StockCode": purchase["StockCode"],
            "Description": purchase["Description"],
            "Customer ID": purchase["Customer ID"]
        })

cancellation_matches = pd.DataFrame(matches)

display(cancellation_matches)

,Purchase_Invoice,Purchase_Quantity,Purchase_Revenue,Purchase_Date,Cancellation_Invoice,Cancellation_Quantity,Cancellation_Date,StockCode,Description,Customer ID
0,581483,80995,168469.6,2011-12-09 09:15:00,C581484,-80995,2011-12-09 09:27:00,23843,"PAPER CRAFT , LITTLE BIRDIE",16446
1,541431,74215,77183.6,2011-01-18 10:01:00,C541433,-74215,2011-01-18 10:17:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,12346


In [30]:
# Finding all purchase lines that have an exact cancellation match
# Same customer + product + opposite quantity + same price

purchases = customer_purchases.copy()

cancellations = df[
    df["Is_Cancellation"] &
    df["Customer ID"].notna()
].copy()

cancellation_keys = cancellations[
    [
        "Customer ID",
        "StockCode",
        "Quantity",
        "Price"
    ]
].copy()

cancellation_keys["Quantity"] = -cancellation_keys["Quantity"]

matched_purchases = purchases.merge(
    cancellation_keys.drop_duplicates(),
    on=["Customer ID", "StockCode", "Quantity", "Price"],
    how="inner"
)

print("Purchase lines with matching cancellation:", len(matched_purchases))
print(
    "Revenue represented by these purchase lines: "
    f"£{matched_purchases['Revenue'].sum():,.2f}"
)

display(
    matched_purchases[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Revenue",
            "Customer ID",
            "InvoiceDate"
        ]
    ]
    .sort_values("Revenue", ascending=False)
    .head(20)
)

Purchase lines with matching cancellation: 1629
Revenue represented by these purchase lines: £207,443.90


,Invoice,StockCode,Description,Quantity,Price,Revenue,Customer ID,InvoiceDate
345,502263,M,Manual,1,10953.50,10953.50,12918,2010-03-23 15:22:00
808,524159,M,Manual,1,10468.80,10468.80,14063,2010-09-27 16:12:00
198,496115,M,Manual,1,8985.60,8985.60,17949,2010-01-29 11:04:00
1220,551697,POST,POSTAGE,1,8142.75,8142.75,16029,2011-05-03 13:46:00
330,501768,M,Manual,1,6958.17,6958.17,15760,2010-03-19 11:45:00
328,501766,M,Manual,1,6958.17,6958.17,15760,2010-03-19 11:35:00
1059,540815,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1930,2.55,4921.50,15749,2011-01-11 12:55:00
1523,571751,M,Manual,1,3949.32,3949.32,12744,2011-10-19 11:18:00
1472,569382,M,Manual,1,3155.95,3155.95,15502,2011-10-03 16:44:00
656,515296,84078A,SET/4 WHITE RETRO STORAGE CUBES,85,34.95,2970.75,13734,2010-07-09 15:20:00


In [ ]:
# Checking how soon matching cancellations occur after the purchase

purchase_candidates = customer_purchases[
    customer_purchases["Invoice"].isin(
        matched_purchases["Invoice"]
    )
].copy()

purchase_candidates = purchase_candidates.merge(
    cancellations[
        [
            "Customer ID",
            "StockCode",
            "Quantity",
            "Price",
            "InvoiceDate",
            "Invoice"
        ]
    ].rename(columns={
        "Quantity": "Cancel_Quantity",
        "InvoiceDate": "Cancel_Date",
        "Invoice": "Cancel_Invoice"
    }),
    left_on=["Customer ID", "StockCode", "Quantity", "Price"],
    right_on=[
        "Customer ID",
        "StockCode",
        "Cancel_Quantity",
        "Price"
    ],
    how="inner"
)

# Keeping cancellations that are after the purchase
purchase_candidates = purchase_candidates[
    purchase_candidates["Cancel_Quantity"] == -purchase_candidates["Quantity"]
].copy()

purchase_candidates["Time_Difference"] = (
    purchase_candidates["Cancel_Date"] -
    purchase_candidates["InvoiceDate"]
)

# Keeping only cancellations occurring after the purchase
purchase_candidates = purchase_candidates[
    purchase_candidates["Time_Difference"] >= pd.Timedelta(0)
].copy()

print("Candidate purchase-cancellation pairs:", len(purchase_candidates))

display(
    purchase_candidates[
        [
            "Invoice",
            "StockCode",
            "Quantity",
            "Price",
            "InvoiceDate",
            "Cancel_Invoice",
            "Cancel_Date",
            "Time_Difference",
            "Customer ID"
        ]
    ]
    .sort_values("Time_Difference")
    .head(30)
)

Candidate purchase-cancellation pairs: 0


,Invoice,StockCode,Quantity,Price,InvoiceDate,Cancel_Invoice,Cancel_Date,Time_Difference,Customer ID


In [32]:
# Correctly matching purchases to cancellation/return lines

purchases = customer_purchases.copy()

cancellations = df[
    df["Is_Cancellation"] &
    df["Customer ID"].notna()
].copy()

# Convert cancellation quantity to positive magnitude
cancellations["Cancelled_Quantity"] = (
    -cancellations["Quantity"]
)

# Match on customer, product, quantity and price
purchase_candidates = purchases.merge(
    cancellations[
        [
            "Customer ID",
            "StockCode",
            "Cancelled_Quantity",
            "Price",
            "InvoiceDate",
            "Invoice"
        ]
    ].rename(columns={
        "InvoiceDate": "Cancel_Date",
        "Invoice": "Cancel_Invoice"
    }),
    left_on=[
        "Customer ID",
        "StockCode",
        "Quantity",
        "Price"
    ],
    right_on=[
        "Customer ID",
        "StockCode",
        "Cancelled_Quantity",
        "Price"
    ],
    how="inner"
)

# Keep only cancellations that happened after the purchase
purchase_candidates["Time_Difference"] = (
    purchase_candidates["Cancel_Date"]
    - purchase_candidates["InvoiceDate"]
)

purchase_candidates = purchase_candidates[
    purchase_candidates["Time_Difference"] >= pd.Timedelta(0)
].copy()

print("Candidate purchase-cancellation pairs:", len(purchase_candidates))

display(
    purchase_candidates[
        [
            "Invoice",
            "StockCode",
            "Quantity",
            "Price",
            "Revenue",
            "InvoiceDate",
            "Cancel_Invoice",
            "Cancel_Date",
            "Time_Difference",
            "Customer ID"
        ]
    ]
    .sort_values("Time_Difference")
    .head(30)
)

Candidate purchase-cancellation pairs: 1246


,Invoice,StockCode,Quantity,Price,Revenue,InvoiceDate,Cancel_Invoice,Cancel_Date,Time_Difference,Customer ID
577,508382,47590B,3,5.45,16.35,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
575,508382,85099B,10,1.95,19.50,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
576,508382,47590A,3,5.45,16.35,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
578,508382,47566B,6,4.65,27.90,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
579,508382,48173C,2,7.49,14.98,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
580,508382,85232B,6,4.95,29.70,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
581,508382,85049G,3,1.25,3.75,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
574,508382,85099C,10,1.95,19.50,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
801,520750,M,1,1868.56,1868.56,2010-08-31 11:04:00,C520749,2010-08-31 11:04:00,0 days 00:00:00,15480
573,508382,85099F,10,1.95,19.50,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634


In [33]:
# Checking for ambiguous purchase-cancellation matches

match_counts = (
    purchase_candidates
    .groupby(
        ["Invoice", "StockCode", "Quantity", "Price", "Customer ID"],
        dropna=False
    )
    .size()
    .reset_index(name="Cancellation_Match_Count")
)

print("Total matched purchase groups:", len(match_counts))

print(
    "Purchase groups with exactly 1 cancellation match:",
    (match_counts["Cancellation_Match_Count"] == 1).sum()
)

print(
    "Purchase groups with multiple cancellation matches:",
    (match_counts["Cancellation_Match_Count"] > 1).sum()
)

display(
    match_counts[
        match_counts["Cancellation_Match_Count"] > 1
    ]
    .sort_values("Cancellation_Match_Count", ascending=False)
    .head(20)
)

Total matched purchase groups: 1190
Purchase groups with exactly 1 cancellation match: 1138
Purchase groups with multiple cancellation matches: 52


,Invoice,StockCode,Quantity,Price,Customer ID,Cancellation_Match_Count
375,504687,M,1,94.80,15354,3
376,504688,M,1,94.80,15354,3
374,504686,M,1,94.80,15354,3
41,491061,85232B,3,4.95,13672,3
40,491061,79323P,4,6.75,13672,2
37,490990,85135A,1,7.95,14606,2
12,490302,47594B,1,1.95,17841,2
4,489594,84507B,1,2.55,15005,2
109,493897,M,1,780.52,14911,2
50,491435,85135A,1,7.95,14606,2


In [34]:
# Inspecting ambiguous purchase-cancellation matches

ambiguous_keys = match_counts[
    match_counts["Cancellation_Match_Count"] > 1
][
    ["Invoice", "StockCode", "Quantity", "Price", "Customer ID"]
]

ambiguous_details = purchase_candidates.merge(
    ambiguous_keys,
    on=["Invoice", "StockCode", "Quantity", "Price", "Customer ID"],
    how="inner"
)

display(
    ambiguous_details[
        [
            "Invoice",
            "StockCode",
            "Quantity",
            "Price",
            "Revenue",
            "InvoiceDate",
            "Cancel_Invoice",
            "Cancel_Date",
            "Time_Difference",
            "Customer ID"
        ]
    ]
    .sort_values(
        ["Invoice", "StockCode", "Time_Difference"]
    )
    .head(50)
)

,Invoice,StockCode,Quantity,Price,Revenue,InvoiceDate,Cancel_Invoice,Cancel_Date,Time_Difference,Customer ID
0,489594,84507B,1,2.55,2.55,2009-12-01 14:19:00,C497621,2010-02-11 12:23:00,71 days 22:04:00,15005
1,489594,84507B,1,2.55,2.55,2009-12-01 14:19:00,C536015,2010-11-29 15:32:00,363 days 01:13:00,15005
2,490302,47594B,1,1.95,1.95,2009-12-04 14:29:00,C545189,2011-02-28 15:19:00,451 days 00:50:00,17841
3,490302,47594B,1,1.95,1.95,2009-12-04 14:29:00,C548995,2011-04-05 14:08:00,486 days 23:39:00,17841
4,490990,85135A,1,7.95,7.95,2009-12-08 17:10:00,C495525,2010-01-25 14:37:00,47 days 21:27:00,14606
5,490990,85135A,1,7.95,7.95,2009-12-08 17:10:00,C495943,2010-01-27 16:14:00,49 days 23:04:00,14606
9,491061,79323P,4,6.75,27.00,2009-12-09 11:40:00,C540634,2011-01-10 12:02:00,397 days 00:22:00,13672
10,491061,79323P,4,6.75,27.00,2009-12-09 11:40:00,C543744,2011-02-11 13:43:00,429 days 02:03:00,13672
6,491061,85232B,3,4.95,14.85,2009-12-09 11:40:00,C540634,2011-01-10 12:02:00,397 days 00:22:00,13672
7,491061,85232B,3,4.95,14.85,2009-12-09 11:40:00,C543744,2011-02-11 13:43:00,429 days 02:03:00,13672


In [36]:


candidates = purchase_candidates.copy()

# Calculate absolute time difference
candidates["Time_Difference_Seconds"] = (
    candidates["Time_Difference"].dt.total_seconds()
)

# For each purchase, keep the closest cancellation
best_matches = (
    candidates
    .sort_values("Time_Difference_Seconds")
    .drop_duplicates(
        subset=["Invoice", "StockCode", "Quantity", "Price", "Customer ID"],
        keep="first"
    )
    .copy()
)

print("One-to-one matched purchase lines:", len(best_matches))

print(
    "Revenue represented by matched purchases: "
    f"£{best_matches['Revenue'].sum():,.2f}"
)

display(
    best_matches[
        [
            "Invoice",
            "StockCode",
            "Quantity",
            "Price",
            "Revenue",
            "InvoiceDate",
            "Cancel_Invoice",
            "Cancel_Date",
            "Time_Difference",
            "Customer ID"
        ]
    ]
    .sort_values("Time_Difference")
    .head(30)
)

One-to-one matched purchase lines: 1190
Revenue represented by matched purchases: £147,624.25


,Invoice,StockCode,Quantity,Price,Revenue,InvoiceDate,Cancel_Invoice,Cancel_Date,Time_Difference,Customer ID
801,520750,M,1,1868.56,1868.56,2010-08-31 11:04:00,C520749,2010-08-31 11:04:00,0 days 00:00:00,15480
580,508382,85232B,6,4.95,29.70,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
575,508382,85099B,10,1.95,19.50,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
576,508382,47590A,3,5.45,16.35,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
577,508382,47590B,3,5.45,16.35,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
581,508382,85049G,3,1.25,3.75,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
579,508382,48173C,2,7.49,14.98,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
573,508382,85099F,10,1.95,19.50,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
574,508382,85099C,10,1.95,19.50,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634
578,508382,47566B,6,4.65,27.90,2010-05-14 14:18:00,C508384,2010-05-14 14:18:00,0 days 00:00:00,13634


In [37]:
# Verifying that the two extreme purchases are included in the matched set

display(
    best_matches[
        best_matches["Invoice"].isin(["581483", "541431"])
    ][
        [
            "Invoice",
            "StockCode",
            "Quantity",
            "Price",
            "Revenue",
            "InvoiceDate",
            "Cancel_Invoice",
            "Cancel_Date",
            "Time_Difference",
            "Customer ID"
        ]
    ]
)

,Invoice,StockCode,Quantity,Price,Revenue,InvoiceDate,Cancel_Invoice,Cancel_Date,Time_Difference,Customer ID


In [38]:
# Verifying the two extreme purchases and their cancellation records directly

extreme_invoices = ["581483", "541431"]

print("PURCHASES:")
display(
    customer_purchases[
        customer_purchases["Invoice"].astype(str).isin(extreme_invoices)
    ][
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Revenue",
            "Customer ID",
            "InvoiceDate"
        ]
    ]
)

print("\nCANCELLATIONS:")
display(
    df[
        df["Invoice"].astype(str).isin(["C581484", "C541433"])
    ][
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Customer ID",
            "InvoiceDate"
        ]
    ]
)

PURCHASES:


,Invoice,StockCode,Description,Quantity,Price,Revenue,Customer ID,InvoiceDate
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.6,12346,2011-01-18 10:01:00
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.6,16446,2011-12-09 09:15:00



CANCELLATIONS:


,Invoice,StockCode,Description,Quantity,Price,Customer ID,InvoiceDate
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346.0,2011-01-18 10:17:00
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,16446.0,2011-12-09 09:27:00


In [39]:
# Measure cancellation/return impact

cancellation_data = df[
    (df["Is_Cancellation"]) &
    (df["Customer ID"].notna())
].copy()

cancellation_data["Return_Revenue"] = (
    cancellation_data["Quantity"] *
    cancellation_data["Price"]
)

print("Cancellation/return lines:", len(cancellation_data))
print("Unique cancellation invoices:", cancellation_data["Invoice"].nunique())
print(
    "Cancellation/return revenue: "
    f"£{cancellation_data['Return_Revenue'].sum():,.2f}"
)

print(
    "Purchase revenue: "
    f"£{customer_purchases['Revenue'].sum():,.2f}"
)

net_revenue = (
    customer_purchases["Revenue"].sum()
    + cancellation_data["Return_Revenue"].sum()
)

print(
    "Net revenue after cancellations/returns: "
    f"£{net_revenue:,.2f}"
)

Cancellation/return lines: 18744
Unique cancellation invoices: 7901
Cancellation/return revenue: £-1,095,136.79
Purchase revenue: £17,374,804.27
Net revenue after cancellations/returns: £16,279,667.48


In [40]:
# Createing final cleaned purchase dataset

final_purchases = customer_purchases.copy()

# Keep only columns needed for analysis
final_purchases = final_purchases[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country",
        "Revenue"
    ]
].copy()

print("Final purchase dataset shape:", final_purchases.shape)
print("Unique customers:", final_purchases["Customer ID"].nunique())
print("Unique invoices:", final_purchases["Invoice"].nunique())
print("Total revenue:", f"£{final_purchases['Revenue'].sum():,.2f}")

print("\nMissing values:")
print(final_purchases.isna().sum())

print("\nDuplicate rows:", final_purchases.duplicated().sum())
print("Negative quantities:", (final_purchases["Quantity"] < 0).sum())
print("Non-positive prices:", (final_purchases["Price"] <= 0).sum())

Final purchase dataset shape: (779425, 9)
Unique customers: 5878
Unique invoices: 36969
Total revenue: £17,374,804.27

Missing values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
dtype: int64

Duplicate rows: 0
Negative quantities: 0
Non-positive prices: 0


In [41]:
# Saveing final cleaned purchase dataset

output_path = "../cleaned_data/customer_purchases_clean.csv"

final_purchases.to_csv(
    output_path,
    index=False
)

print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: ../cleaned_data/customer_purchases_clean.csv


In [42]:
# Createing separate cancellation/return dataset

cancellation_returns = df[
    (df["Is_Cancellation"]) &
    (df["Customer ID"].notna())
].copy()

cancellation_returns["Return_Revenue"] = (
    cancellation_returns["Quantity"] *
    cancellation_returns["Price"]
)

cancellation_returns = cancellation_returns[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country",
        "Return_Revenue"
    ]
].copy()

print("Cancellation/return dataset shape:", cancellation_returns.shape)
print("Unique customers:", cancellation_returns["Customer ID"].nunique())
print("Unique cancellation invoices:", cancellation_returns["Invoice"].nunique())
print(
    "Return/cancellation value:",
    f"£{cancellation_returns['Return_Revenue'].sum():,.2f}"
)

print("\nMissing values:")
print(cancellation_returns.isna().sum())

print("\nDuplicate rows:", cancellation_returns.duplicated().sum())

Cancellation/return dataset shape: (18744, 9)
Unique customers: 2572
Unique cancellation invoices: 7901
Return/cancellation value: £-1,095,136.79

Missing values:
Invoice           0
StockCode         0
Description       0
Quantity          0
InvoiceDate       0
Price             0
Customer ID       0
Country           0
Return_Revenue    0
dtype: int64

Duplicate rows: 354


In [43]:
# Removeing exact duplicate cancellation/return records

before_returns = len(cancellation_returns)

cancellation_returns = (
    cancellation_returns
    .drop_duplicates(keep="first")
    .copy()
)

after_returns = len(cancellation_returns)

print("Rows before deduplication:", before_returns)
print("Duplicate rows removed:", before_returns - after_returns)
print("Rows after deduplication:", after_returns)
print("Remaining duplicates:", cancellation_returns.duplicated().sum())

print(
    "Return/cancellation value after deduplication:",
    f"£{cancellation_returns['Return_Revenue'].sum():,.2f}"
)

Rows before deduplication: 18744
Duplicate rows removed: 354
Rows after deduplication: 18390
Remaining duplicates: 0
Return/cancellation value after deduplication: £-1,084,812.98


In [44]:
# Saveing final cancellation/return dataset

output_path = "../cleaned_data/cancellation_returns_clean.csv"

cancellation_returns.to_csv(
    output_path,
    index=False
)

print(f"Cancellation/return dataset saved to: {output_path}")

Cancellation/return dataset saved to: ../cleaned_data/cancellation_returns_clean.csv


In [45]:
# Createing customer-level purchasing metrics

customer_metrics = (
    final_purchases
    .groupby("Customer ID")
    .agg(
        First_Purchase_Date=("InvoiceDate", "min"),
        Last_Purchase_Date=("InvoiceDate", "max"),
        Total_Orders=("Invoice", "nunique"),
        Total_Items=("Quantity", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Average_Order_Value=("Revenue", lambda x: x.sum() / final_purchases.loc[x.index, "Invoice"].nunique())
    )
    .reset_index()
)

# Calculate purchase frequency
customer_metrics["Purchase_Frequency"] = (
    customer_metrics["Total_Orders"] /
    (
        (customer_metrics["Last_Purchase_Date"] -
         customer_metrics["First_Purchase_Date"])
        .dt.days
        .clip(lower=1)
        / 30
    )
)

print("Customer-level dataset shape:", customer_metrics.shape)

display(customer_metrics.head(10))

Customer-level dataset shape: (5878, 8)


,Customer ID,First_Purchase_Date,Last_Purchase_Date,Total_Orders,Total_Items,Total_Revenue,Average_Order_Value,Purchase_Frequency
0,12346,2009-12-14 08:34:00,2011-01-18 10:01:00,12,74285,77556.46,6463.038333,0.900000
1,12347,2010-10-31 14:20:00,2011-12-07 15:52:00,8,2967,4921.53,615.191250,0.597015
2,12348,2010-09-27 14:59:00,2011-09-25 13:13:00,5,2714,2019.40,403.880000,0.414365
3,12349,2010-04-29 13:20:00,2011-11-21 09:51:00,4,1624,4428.69,1107.172500,0.210526
4,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,1,197,334.40,334.400000,30.000000
5,12351,2010-11-29 15:23:00,2010-11-29 15:23:00,1,261,300.93,300.930000,30.000000
6,12352,2010-11-12 10:20:00,2011-11-03 14:37:00,10,724,2849.84,284.984000,0.842697
7,12353,2010-10-27 12:44:00,2011-05-19 17:47:00,2,212,406.76,203.380000,0.294118
8,12354,2011-04-21 13:11:00,2011-04-21 13:11:00,1,530,1079.40,1079.400000,30.000000
9,12355,2010-05-21 11:59:00,2011-05-09 13:49:00,2,543,947.61,473.805000,0.169972


In [46]:
# Calculateing customer recency

analysis_date = final_purchases["InvoiceDate"].max()

customer_metrics["Recency_Days"] = (
    analysis_date - customer_metrics["Last_Purchase_Date"]
).dt.days

print("Analysis date:", analysis_date)
print("\nRecency statistics:")
print(customer_metrics["Recency_Days"].describe())

display(
    customer_metrics[
        [
            "Customer ID",
            "Last_Purchase_Date",
            "Recency_Days"
        ]
    ].head(10)
)

Analysis date: 2011-12-09 12:50:00

Recency statistics:
count    5878.000000
mean      200.331916
std       209.338707
min         0.000000
25%        25.000000
50%        95.000000
75%       379.000000
max       738.000000
Name: Recency_Days, dtype: float64


,Customer ID,Last_Purchase_Date,Recency_Days
0,12346,2011-01-18 10:01:00,325
1,12347,2011-12-07 15:52:00,1
2,12348,2011-09-25 13:13:00,74
3,12349,2011-11-21 09:51:00,18
4,12350,2011-02-02 16:01:00,309
5,12351,2010-11-29 15:23:00,374
6,12352,2011-11-03 14:37:00,35
7,12353,2011-05-19 17:47:00,203
8,12354,2011-04-21 13:11:00,231
9,12355,2011-05-09 13:49:00,213


In [47]:
# Inspecting customer recency distribution

recency_bins = [0, 30, 60, 90, 120, 180, 270, 365, 540, 9999]
recency_labels = [
    "0-30 days",
    "31-60 days",
    "61-90 days",
    "91-120 days",
    "121-180 days",
    "181-270 days",
    "271-365 days",
    "366-540 days",
    "541+ days"
]

customer_metrics["Recency_Band"] = pd.cut(
    customer_metrics["Recency_Days"],
    bins=recency_bins,
    labels=recency_labels,
    include_lowest=True
)

recency_distribution = (
    customer_metrics["Recency_Band"]
    .value_counts()
    .sort_index()
    .to_frame("Customers")
)

recency_distribution["Percentage"] = (
    recency_distribution["Customers"]
    / len(customer_metrics)
    * 100
)

display(recency_distribution)

,Customers,Percentage
Recency_Band,,
0-30 days,1694,28.819326
31-60 days,728,12.385165
61-90 days,471,8.012930
91-120 days,230,3.912896
121-180 days,357,6.073494
181-270 days,474,8.063967
271-365 days,326,5.546104
366-540 days,1035,17.608030
541+ days,563,9.578088


In [48]:
# Calculateing purchase intervals for repeat customers

purchase_dates = (
    customer_purchases[
        ["Customer ID", "InvoiceDate"]
    ]
    .drop_duplicates()
    .sort_values(["Customer ID", "InvoiceDate"])
)

purchase_dates["Previous_Purchase_Date"] = (
    purchase_dates
    .groupby("Customer ID")["InvoiceDate"]
    .shift(1)
)

purchase_dates["Days_Between_Purchases"] = (
    purchase_dates["InvoiceDate"]
    - purchase_dates["Previous_Purchase_Date"]
).dt.days

purchase_gaps = purchase_dates.dropna(
    subset=["Days_Between_Purchases"]
)

print("Repeat purchase intervals:", len(purchase_gaps))
print("Customers with repeat purchases:",
      purchase_gaps["Customer ID"].nunique())

print("\nPurchase interval statistics:")
print(
    purchase_gaps["Days_Between_Purchases"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Repeat purchase intervals: 30973
Customers with repeat purchases: 4256

Purchase interval statistics:
count    30973.000000
mean        51.434637
std         75.832387
min          0.000000
25%          7.000000
50%         24.000000
75%         61.000000
90%        134.800000
95%        206.000000
99%        369.000000
max        714.000000
Name: Days_Between_Purchases, dtype: float64


In [49]:
# Compareing potential churn thresholds

thresholds = [60, 90, 120, 180, 270, 365]

churn_threshold_analysis = []

for threshold in thresholds:
    churned = (customer_metrics["Recency_Days"] > threshold).sum()
    percentage = churned / len(customer_metrics) * 100

    churn_threshold_analysis.append({
        "Threshold_Days": threshold,
        "Churned_Customers": churned,
        "Churn_Rate_Percent": percentage
    })

churn_threshold_analysis = pd.DataFrame(churn_threshold_analysis)

display(churn_threshold_analysis)

,Threshold_Days,Churned_Customers,Churn_Rate_Percent
0,60,3456,58.795509
1,90,2985,50.782579
2,120,2755,46.869684
3,180,2398,40.796189
4,270,1924,32.732222
5,365,1598,27.186118


In [50]:
# Createing customer churn flag

CHURN_THRESHOLD_DAYS = 180

customer_metrics["Churn_Flag"] = (
    customer_metrics["Recency_Days"] > CHURN_THRESHOLD_DAYS
).astype(int)

customer_metrics["Churn_Status"] = np.where(
    customer_metrics["Churn_Flag"] == 1,
    "Churned",
    "Active"
)

print("Churn threshold:", CHURN_THRESHOLD_DAYS, "days")
print("\nChurn status:")
print(customer_metrics["Churn_Status"].value_counts())

print("\nChurn percentage:")
print(
    customer_metrics["Churn_Flag"].mean() * 100
)

display(
    customer_metrics[
        [
            "Customer ID",
            "Recency_Days",
            "Churn_Flag",
            "Churn_Status"
        ]
    ].head(10)
)

Churn threshold: 180 days

Churn status:
Churn_Status
Active     3480
Churned    2398
Name: count, dtype: int64

Churn percentage:
40.7961891799932


,Customer ID,Recency_Days,Churn_Flag,Churn_Status
0,12346,325,1,Churned
1,12347,1,0,Active
2,12348,74,0,Active
3,12349,18,0,Active
4,12350,309,1,Churned
5,12351,374,1,Churned
6,12352,35,0,Active
7,12353,203,1,Churned
8,12354,231,1,Churned
9,12355,213,1,Churned


In [51]:
# Calculateing customer lifetime

customer_metrics["Customer_Lifetime_Days"] = (
    customer_metrics["Last_Purchase_Date"]
    - customer_metrics["First_Purchase_Date"]
).dt.days

print("Customer lifetime statistics:")
print(
    customer_metrics["Customer_Lifetime_Days"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nOne-time customers:")
print(
    (customer_metrics["Total_Orders"] == 1).sum()
)

print("\nRepeat customers:")
print(
    (customer_metrics["Total_Orders"] > 1).sum()
)

Customer lifetime statistics:
count    5878.000000
mean      273.022457
std       258.807591
min         0.000000
25%         0.000000
50%       220.500000
75%       511.000000
90%       667.000000
95%       712.000000
99%       732.000000
max       738.000000
Name: Customer_Lifetime_Days, dtype: float64

One-time customers:
1623

Repeat customers:
4255


In [52]:
# Compareing churn between one-time and repeat customers

customer_metrics["Customer_Type"] = np.where(
    customer_metrics["Total_Orders"] == 1,
    "One-Time",
    "Repeat"
)

customer_type_churn = (
    customer_metrics
    .groupby("Customer_Type")
    .agg(
        Customers=("Customer ID", "count"),
        Churned_Customers=("Churn_Flag", "sum"),
        Churn_Rate=("Churn_Flag", "mean"),
        Average_Orders=("Total_Orders", "mean"),
        Average_Revenue=("Total_Revenue", "mean")
    )
    .reset_index()
)

customer_type_churn["Churn_Rate"] *= 100

display(customer_type_churn)

,Customer_Type,Customers,Churned_Customers,Churn_Rate,Average_Orders,Average_Revenue
0,One-Time,1623,1152,70.979667,1.000000,345.207986
1,Repeat,4255,1246,29.283196,8.306933,3951.711329


In [53]:
# Inspecting customer revenue distribution

revenue_stats = customer_metrics["Total_Revenue"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

print("Customer revenue statistics:")
print(revenue_stats)

print("\nRevenue contribution by customer segment:")

customer_revenue_segments = pd.DataFrame({
    "Revenue_Segment": pd.cut(
        customer_metrics["Total_Revenue"],
        bins=[0, 500, 1000, 5000, 10000, float("inf")],
        labels=[
            "£0-£500",
            "£501-£1,000",
            "£1,001-£5,000",
            "£5,001-£10,000",
            "£10,000+"
        ]
    )
}).groupby("Revenue_Segment", observed=False).agg(
    Customers=("Revenue_Segment", "count")
).reset_index()

customer_revenue_segments["Customer_Percentage"] = (
    customer_revenue_segments["Customers"]
    / len(customer_metrics)
    * 100
)

display(customer_revenue_segments)

Customer revenue statistics:
count      5878.000000
mean       2955.904095
std       14440.852688
min           2.950000
25%         342.280000
50%         867.740000
75%        2248.305000
90%        5465.741000
95%        9374.227000
99%       29205.901000
max      580987.040000
Name: Total_Revenue, dtype: float64

Revenue contribution by customer segment:


,Revenue_Segment,Customers,Customer_Percentage
0,£0-£500,2066,35.148010
1,"£501-£1,000",1107,18.832936
2,"£1,001-£5,000",2060,35.045934
3,"£5,001-£10,000",384,6.532834
4,"£10,000+",261,4.440286


In [54]:
# Identifying high-value customers

high_value_customers = (
    customer_metrics
    .sort_values("Total_Revenue", ascending=False)
    [[
        "Customer ID",
        "Total_Orders",
        "Total_Items",
        "Total_Revenue",
        "Average_Order_Value",
        "Purchase_Frequency",
        "Recency_Days",
        "Churn_Flag",
        "Churn_Status"
    ]]
    .head(20)
)

display(high_value_customers)

,Customer ID,Total_Orders,Total_Items,Total_Revenue,Average_Order_Value,Purchase_Frequency,Recency_Days,Churn_Flag,Churn_Status
5692,18102,145,181645,580987.04,4006.807172,5.894309,0,0,Active
2277,14646,151,367193,528602.52,3500.678940,6.163265,1,0,Active
1789,14156,156,164325,313437.62,2009.215513,6.428571,9,0,Active
2538,14911,398,147972,291420.81,732.213090,16.200814,0,0,Active
5050,17450,51,83914,244784.25,4799.691176,3.566434,7,0,Active
1331,13694,143,188201,195640.69,1368.116713,5.868673,3,0,Active
5109,17511,60,117174,172132.87,2868.881167,2.452316,2,0,Active
4061,16446,2,80997,168472.50,84236.250000,0.294118,0,0,Active
4295,16684,55,104810,147142.77,2675.323091,2.266484,3,0,Active
68,12415,28,91447,144458.37,5159.227500,1.669980,23,0,Active


In [55]:
# Compareing revenue and purchasing behavior of active vs churned customers

churn_comparison = (
    customer_metrics
    .groupby("Churn_Status")
    .agg(
        Customers=("Customer ID", "count"),
        Average_Revenue=("Total_Revenue", "mean"),
        Median_Revenue=("Total_Revenue", "median"),
        Average_Orders=("Total_Orders", "mean"),
        Median_Orders=("Total_Orders", "median"),
        Average_Order_Value=("Average_Order_Value", "mean"),
        Average_Recency=("Recency_Days", "mean")
    )
    .reset_index()
)

display(churn_comparison)

,Churn_Status,Customers,Average_Revenue,Median_Revenue,Average_Orders,Median_Orders,Average_Order_Value,Average_Recency
0,Active,3480,4321.736447,1485.135,8.842241,5.0,409.857956,47.949138
1,Churned,2398,973.795427,408.445,2.584654,2.0,349.369181,421.471226


In [56]:
# Churn rate by purchase frequency

frequency_churn = (
    customer_metrics
    .groupby("Total_Orders")
    .agg(
        Customers=("Customer ID", "count"),
        Churned_Customers=("Churn_Flag", "sum")
    )
    .reset_index()
)

frequency_churn["Churn_Rate_Percent"] = (
    frequency_churn["Churned_Customers"]
    / frequency_churn["Customers"]
    * 100
)

display(frequency_churn.head(20))

,Total_Orders,Customers,Churned_Customers,Churn_Rate_Percent
0,1,1623,1152,70.979667
1,2,944,506,53.601695
2,3,664,262,39.457831
3,4,486,174,35.802469
4,5,360,95,26.388889
5,6,280,69,24.642857
6,7,222,44,19.819820
7,8,172,28,16.279070
8,9,153,9,5.882353
9,10,98,9,9.183673


In [57]:
# Createing customer purchase-frequency segments

customer_metrics["Frequency_Segment"] = pd.cut(
    customer_metrics["Total_Orders"],
    bins=[0, 1, 2, 4, 7, 12, float("inf")],
    labels=[
        "1 Order",
        "2 Orders",
        "3-4 Orders",
        "5-7 Orders",
        "8-12 Orders",
        "13+ Orders"
    ]
)

frequency_segment_churn = (
    customer_metrics
    .groupby("Frequency_Segment", observed=False)
    .agg(
        Customers=("Customer ID", "count"),
        Churned_Customers=("Churn_Flag", "sum"),
        Average_Revenue=("Total_Revenue", "mean"),
        Average_Recency=("Recency_Days", "mean")
    )
    .reset_index()
)

frequency_segment_churn["Churn_Rate_Percent"] = (
    frequency_segment_churn["Churned_Customers"]
    / frequency_segment_churn["Customers"]
    * 100
)

display(frequency_segment_churn)

,Frequency_Segment,Customers,Churned_Customers,Average_Revenue,Average_Recency,Churn_Rate_Percent
0,1 Order,1623,1152,345.207986,354.160197,70.979667
1,2 Orders,944,506,859.570819,244.165254,53.601695
2,3-4 Orders,1150,436,1231.598714,174.761739,37.913043
3,5-7 Orders,862,208,2121.633646,117.409513,24.129930
4,8-12 Orders,650,68,3896.752697,70.852308,10.461538
5,13+ Orders,649,28,15755.039871,37.010786,4.314330


In [58]:
# Churn rate by recency segment

recency_churn = (
    customer_metrics
    .groupby("Recency_Band", observed=False)
    .agg(
        Customers=("Customer ID", "count"),
        Churned_Customers=("Churn_Flag", "sum"),
        Average_Revenue=("Total_Revenue", "mean")
    )
    .reset_index()
)

recency_churn["Churn_Rate_Percent"] = (
    recency_churn["Churned_Customers"]
    / recency_churn["Customers"]
    * 100
)

display(recency_churn)

,Recency_Band,Customers,Churned_Customers,Average_Revenue,Churn_Rate_Percent
0,0-30 days,1694,0,6570.497901,0.0
1,31-60 days,728,0,2592.984302,0.0
2,61-90 days,471,0,2063.165909,0.0
3,91-120 days,230,0,2021.961217,0.0
4,121-180 days,357,0,1637.884017,0.0
5,181-270 days,474,474,1270.624306,100.0
6,271-365 days,326,326,1423.669969,100.0
7,366-540 days,1035,1035,880.543170,100.0
8,541+ days,563,563,634.825794,100.0


In [ ]:
# Assigning each customer their most frequent purchasing country

customer_country = (
    customer_purchases
    .groupby(["Customer ID", "Country"])
    .size()
    .reset_index(name="Purchase_Lines")
    .sort_values(
        ["Customer ID", "Purchase_Lines"],
        ascending=[True, False]
    )
    .drop_duplicates("Customer ID")
    [["Customer ID", "Country"]]
)

# Adding country to the customer-level dataset
customer_metrics = customer_metrics.drop(columns=["Country"], errors="ignore")

customer_metrics = customer_metrics.merge(
    customer_country,
    on="Customer ID",
    how="left"
)

# Calculateing churn by country
country_churn = (
    customer_metrics
    .groupby("Country")
    .agg(
        Customers=("Customer ID", "count"),
        Churned_Customers=("Churn_Flag", "sum"),
        Average_Revenue=("Total_Revenue", "mean")
    )
    .reset_index()
)

country_churn["Churn_Rate_Percent"] = (
    country_churn["Churned_Customers"]
    / country_churn["Customers"]
    * 100
)

# Focusing on countries with at least 20 customers
country_churn = (
    country_churn[country_churn["Customers"] >= 20]
    .sort_values("Churn_Rate_Percent", ascending=False)
)

display(country_churn)

,Country,Customers,Churned_Customers,Average_Revenue,Churn_Rate_Percent
24,Netherlands,22,16,25183.549545,72.727273
28,Portugal,24,10,2314.782500,41.666667
38,United Kingdom,5350,2203,2689.614295,41.177570
3,Belgium,28,10,2330.448214,35.714286
32,Spain,38,13,2807.495263,34.210526
14,Germany,106,28,4008.704349,26.415094
13,France,95,24,3674.814316,25.263158
34,Switzerland,22,5,4693.760909,22.727273


In [64]:
for col in customer_metrics.columns:
    print(col)

Customer ID
First_Purchase_Date
Last_Purchase_Date
Total_Orders
Total_Items
Total_Revenue
Average_Order_Value
Purchase_Frequency
Recency_Days
Recency_Band
Churn_Flag
Churn_Status
Customer_Lifetime_Days
Customer_Type
Frequency_Segment
Country


In [65]:
# Creating customer revenue segments

customer_metrics["Revenue_Segment"] = pd.cut(
    customer_metrics["Total_Revenue"],
    bins=[0, 500, 1000, 5000, 10000, float("inf")],
    labels=[
        "£0-£500",
        "£501-£1,000",
        "£1,001-£5,000",
        "£5,001-£10,000",
        "£10,000+"
    ],
    include_lowest=True
)

print("Revenue segment distribution:")
print(customer_metrics["Revenue_Segment"].value_counts().sort_index())

Revenue segment distribution:
Revenue_Segment
£0-£500           2066
£501-£1,000       1107
£1,001-£5,000     2060
£5,001-£10,000     384
£10,000+           261
Name: count, dtype: int64


In [66]:
# Churn rate by customer revenue segment

revenue_churn = (
    customer_metrics
    .groupby("Revenue_Segment", observed=True)
    .agg(
        Customers=("Customer ID", "count"),
        Churned_Customers=("Churn_Flag", "sum"),
        Average_Orders=("Total_Orders", "mean"),
        Average_Revenue=("Total_Revenue", "mean")
    )
    .reset_index()
)

revenue_churn["Churn_Rate_Percent"] = (
    revenue_churn["Churned_Customers"]
    / revenue_churn["Customers"]
    * 100
)

display(revenue_churn)

,Revenue_Segment,Customers,Churned_Customers,Average_Orders,Average_Revenue,Churn_Rate_Percent
0,£0-£500,2066,1365,1.489351,256.249115,66.069700
1,"£501-£1,000",1107,491,2.976513,727.544527,44.354110
2,"£1,001-£5,000",2060,498,6.805825,2249.806654,24.174757
3,"£5,001-£10,000",384,25,16.486979,7008.357667,6.510417
4,"£10,000+",261,19,39.256705,33387.704038,7.279693


In [67]:
# Checking country consistency at customer level

country_counts = (
    customer_purchases
    .groupby("Customer ID")["Country"]
    .nunique()
)

print("Customers with multiple countries:", (country_counts > 1).sum())
print("Customers with exactly one country:", (country_counts == 1).sum())

Customers with multiple countries: 13
Customers with exactly one country: 5865


In [68]:
# Final customer-level data quality check

print("Customer-level dataset shape:", customer_metrics.shape)

print("\nMissing values:")
print(customer_metrics.isna().sum())

print("\nDuplicate customer IDs:", customer_metrics["Customer ID"].duplicated().sum())

print("\nUnique customers:", customer_metrics["Customer ID"].nunique())

Customer-level dataset shape: (5878, 17)

Missing values:
Customer ID               0
First_Purchase_Date       0
Last_Purchase_Date        0
Total_Orders              0
Total_Items               0
Total_Revenue             0
Average_Order_Value       0
Purchase_Frequency        0
Recency_Days              0
Recency_Band              0
Churn_Flag                0
Churn_Status              0
Customer_Lifetime_Days    0
Customer_Type             0
Frequency_Segment         0
Country                   0
Revenue_Segment           0
dtype: int64

Duplicate customer IDs: 0

Unique customers: 5878


In [ ]:
# Saving final customer-level dataset

customer_metrics.to_csv(
    "../cleaned_data/customer_metrics.csv",
    index=False
)

print("Customer-level dataset saved successfully.")
print("Rows:", len(customer_metrics))
print("Columns:", len(customer_metrics.columns))

Customer-level dataset saved successfully.
Rows: 5878
Columns: 17


In [70]:
# Saving final cleaned purchase transaction dataset

customer_purchases.to_csv(
    "../cleaned_data/customer_purchases_clean.csv",
    index=False
)

print("Purchase transaction dataset saved successfully.")
print("Rows:", len(customer_purchases))
print("Columns:", len(customer_purchases.columns))
print("Unique customers:", customer_purchases["Customer ID"].nunique())
print("Unique invoices:", customer_purchases["Invoice"].nunique())

Purchase transaction dataset saved successfully.
Rows: 779425
Columns: 15
Unique customers: 5878
Unique invoices: 36969


In [71]:
# Final data-quality check for cleaned purchase transactions

print("=== FINAL PURCHASE DATA QUALITY CHECK ===")

print("Rows:", len(customer_purchases))
print("Columns:", len(customer_purchases.columns))
print("Duplicate rows:", customer_purchases.duplicated().sum())
print("Missing values:")
print(customer_purchases.isna().sum())

print("\nUnique customers:", customer_purchases["Customer ID"].nunique())
print("Unique invoices:", customer_purchases["Invoice"].nunique())
print("Unique products:", customer_purchases["StockCode"].nunique())

print("\nNegative quantities:", (customer_purchases["Quantity"] < 0).sum())
print("Non-positive prices:", (customer_purchases["Price"] <= 0).sum())

print("\nDate range:")
print("Start:", customer_purchases["InvoiceDate"].min())
print("End:", customer_purchases["InvoiceDate"].max())

print("\nRevenue:")
print("Total:", round(customer_purchases["Revenue"].sum(), 2))
print("Average transaction line:", round(customer_purchases["Revenue"].mean(), 2))

=== FINAL PURCHASE DATA QUALITY CHECK ===
Rows: 779425
Columns: 15
Duplicate rows: 0
Missing values:
Invoice                 0
StockCode               0
Description             0
Quantity                0
InvoiceDate             0
Price                   0
Customer ID             0
Country                 0
Is_Cancellation         0
Is_Negative_Quantity    0
Is_Zero_Price           0
Is_Negative_Price       0
Has_Customer_ID         0
Transaction_Type        0
Revenue                 0
dtype: int64

Unique customers: 5878
Unique invoices: 36969
Unique products: 4631

Negative quantities: 0
Non-positive prices: 0

Date range:
Start: 2009-12-01 07:45:00
End: 2011-12-09 12:50:00

Revenue:
Total: 17374804.27
Average transaction line: 22.29


In [72]:
# Createing database-ready customer table

customer_db = customer_metrics.copy()

# Convert datetime columns to a MySQL-friendly format
customer_db["First_Purchase_Date"] = (
    customer_db["First_Purchase_Date"].dt.strftime("%Y-%m-%d %H:%M:%S")
)

customer_db["Last_Purchase_Date"] = (
    customer_db["Last_Purchase_Date"].dt.strftime("%Y-%m-%d %H:%M:%S")
)

# Save database-ready customer table
customer_db.to_csv(
    "../cleaned_data/customers.csv",
    index=False
)

print("Database-ready customer table saved successfully.")
print("Rows:", len(customer_db))
print("Columns:", len(customer_db.columns))

Database-ready customer table saved successfully.
Rows: 5878
Columns: 17
